In [1]:
import sqlite3
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

conn = sqlite3.connect("../retailiq.db")
df = pd.read_sql("SELECT * FROM customer_features", conn)
conn.close()

feature_cols = [
    "frequency", "repeat_order_count", "avg_days_between_orders",
    "avg_review_score", "review_count", "avg_delivery_days", "avg_delay_days",
    "treatment_voucher"
]
target = "churned"

model_df = df[feature_cols + [target]].dropna()
X = model_df[feature_cols]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

In [2]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1, 0.2],
}

xgb = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, eval_metric="logloss")
grid_search = GridSearchCV(
    xgb, param_grid, cv=5, scoring="roc_auc", n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV ROC-AUC:", round(grid_search.best_score_, 3))

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best params: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 200}
Best CV ROC-AUC: 0.747


In [3]:
best_clf = grid_search.best_estimator_
probs = best_clf.predict_proba(X_test)[:, 1]
preds = best_clf.predict(X_test)

test_auc = roc_auc_score(y_test, probs)
print(f"Tuned model — Test ROC-AUC: {round(test_auc, 3)}")
print(classification_report(y_test, preds))

baseline_auc = None  # <-- fill in your Day 9 ROC-AUC here
if baseline_auc:
    print(f"\nBaseline ROC-AUC: {baseline_auc}")
    print(f"Improvement: {round(test_auc - baseline_auc, 3)} points")

Tuned model — Test ROC-AUC: 0.751
              precision    recall  f1-score   support

           0       0.34      0.67      0.45      3690
           1       0.89      0.68      0.77     14868

    accuracy                           0.68     18558
   macro avg       0.62      0.68      0.61     18558
weighted avg       0.78      0.68      0.71     18558



In [4]:
import json

metrics_log = {
    "clv_model": {
        "baseline_mae": baseline_mae if 'baseline_mae' in dir() else "fill in",
        "tuned_mae": mae if 'mae' in dir() else "fill in",
    },
    "churn_model": {
        "baseline_auc": baseline_auc if 'baseline_auc' in dir() else "fill in",
        "tuned_auc": round(test_auc, 3),
        "best_params": grid_search.best_params_,
    }
}

with open("../docs/model_metrics.json", "w") as f:
    json.dump(metrics_log, f, indent=2, default=str)

print("Saved metrics log to docs/model_metrics.json")

Saved metrics log to docs/model_metrics.json


In [5]:
import joblib
joblib.dump(best_clf, "../models_churn.pkl")

['../models_churn.pkl']